# YOLO whole-bottle confidence audit

Kaggle-only pass over `bottle-reranker-dataset`. The notebook runs YOLO11x instance segmentation, makes **one whole-bottle crop per image**, and exports the 30 valid crops with the lowest raw YOLO confidence plus the 10 with the highest confidence.

Before running: attach the dataset, enable a GPU, and enable Internet so Ultralytics can download `yolo11x-seg.pt` (or attach the weights and set `MODEL_PATH`). No encoder or local inference is used. Images with no bottle detection are reported separately and are not silently mixed into the crop ranking.

In [ ]:
%pip install -q "ultralytics>=8.3,<9" "pandas>=2.2,<4" "Pillow>=10.4,<13" "opencv-python-headless>=4.10,<5"

In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import shutil
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from PIL import Image, ImageDraw, ImageFont, ImageOps
from tqdm.auto import tqdm
from ultralytics import YOLO

# Change only these values when the Kaggle attachment name differs.
DATASET_HINTS = ("bottle-reranker-dataset", "viscaner-bottle-reranker-data")
DATASET_ROOT_OVERRIDE: str | None = None
IMAGE_ROOT_OVERRIDE: str | None = None
MODEL_PATH = "yolo11x-seg.pt"

IMAGE_SIZE = 1024
INFERENCE_CONFIDENCE_FLOOR = 0.001
IOU_THRESHOLD = 0.70
BOTTLE_CLASS_ID = 39  # COCO: bottle
BATCH_SIZE = 8
PAD_FRACTION = 0.06
LOWEST_N = 30
HIGHEST_N = 10
CHECKPOINT_EVERY = 250

INPUT_BASE = Path("/kaggle/input")
OUTPUT_ROOT = Path("/kaggle/working/yolo_bottle_confidence_audit")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
PREDICTIONS_CSV = OUTPUT_ROOT / "all_predictions.csv"
RUN_METADATA_JSON = OUTPUT_ROOT / "run_metadata.json"

assert torch.cuda.is_available(), "Enable a GPU accelerator in Kaggle before running the audit."
DEVICE = 0
USE_HALF = True
print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
def resolve_dataset_root() -> Path:
    if DATASET_ROOT_OVERRIDE:
        root = Path(DATASET_ROOT_OVERRIDE)
        if not root.is_dir():
            raise FileNotFoundError(root)
        return root

    available = sorted(p for p in INPUT_BASE.iterdir() if p.is_dir())
    matches = [
        p for p in available
        if any(hint.lower() in p.name.lower() for hint in DATASET_HINTS)
    ]
    if len(matches) != 1:
        names = [p.name for p in available]
        raise RuntimeError(
            f"Expected one attached bottle-reranker dataset, found {len(matches)}. "
            f"Set DATASET_ROOT_OVERRIDE explicitly. Available inputs: {names}"
        )
    return matches[0]


def choose_image_root(dataset_root: Path) -> Path:
    if IMAGE_ROOT_OVERRIDE:
        root = Path(IMAGE_ROOT_OVERRIDE)
        if not root.is_dir():
            raise FileNotFoundError(root)
        return root

    # Prefer original query frames. A packaged reranker dataset may expose
    # only queries/bottle, which is still a valid single-bottle audit pool.
    preferred = [
        "queries/source",
        "queries/original",
        "images",
        "data/images",
        "train/images",
        "queries/bottle",
    ]
    for relative in preferred:
        candidate = dataset_root / relative
        if candidate.is_dir() and any(candidate.rglob("*.*")):
            return candidate
    return dataset_root


def discover_images(image_root: Path) -> list[Path]:
    extensions = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff"}
    excluded_parts = {
        "mask", "masks", "masks_npz", "normalized", "top",
        "reference", "references", "audit", "outputs", "__macosx",
    }
    files = []
    for path in image_root.rglob("*"):
        if not path.is_file() or path.suffix.lower() not in extensions:
            continue
        relative_parts = {part.lower() for part in path.relative_to(image_root).parts[:-1]}
        if relative_parts & excluded_parts:
            continue
        files.append(path)
    return sorted(files)


DATASET_ROOT = resolve_dataset_root()
IMAGE_ROOT = choose_image_root(DATASET_ROOT)
IMAGE_PATHS = discover_images(IMAGE_ROOT)
if not IMAGE_PATHS:
    raise RuntimeError(f"No images found under {IMAGE_ROOT}")

print(f"Dataset root: {DATASET_ROOT}")
print(f"Image root:   {IMAGE_ROOT}")
print(f"Images:       {len(IMAGE_PATHS):,}")
print("First files:")
for sample in IMAGE_PATHS[:8]:
    print(" -", sample.relative_to(DATASET_ROOT))

## Load YOLO11x-seg

The ranking uses the confidence of the highest-confidence COCO `bottle` instance in each image. The crop bounds come from that instance mask with 6% padding, so the neck and cap remain inside the single whole-bottle crop.

In [ ]:
model = YOLO(MODEL_PATH)
resolved_weights = Path(str(getattr(model, "ckpt_path", MODEL_PATH)))
weights_sha256 = None
if resolved_weights.is_file():
    digest = hashlib.sha256()
    with resolved_weights.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    weights_sha256 = digest.hexdigest()

run_metadata = {
    "dataset_root": str(DATASET_ROOT),
    "image_root": str(IMAGE_ROOT),
    "image_count": len(IMAGE_PATHS),
    "model_path": MODEL_PATH,
    "weights_sha256": weights_sha256,
    "image_size": IMAGE_SIZE,
    "confidence_floor": INFERENCE_CONFIDENCE_FLOOR,
    "iou_threshold": IOU_THRESHOLD,
    "bottle_class_id": BOTTLE_CLASS_ID,
    "pad_fraction": PAD_FRACTION,
}
if RUN_METADATA_JSON.exists():
    previous_metadata = json.loads(RUN_METADATA_JSON.read_text())
    if previous_metadata != run_metadata:
        raise RuntimeError(
            "Existing checkpoint belongs to different settings. Move or delete "
            f"{OUTPUT_ROOT} before starting a new audit."
        )
else:
    RUN_METADATA_JSON.write_text(json.dumps(run_metadata, indent=2))

print(json.dumps(run_metadata, indent=2))

In [ ]:
def load_rgb(path: Path) -> np.ndarray:
    with Image.open(path) as image:
        return np.asarray(ImageOps.exif_transpose(image).convert("RGB"))


def padded_mask_bbox(result, index: int, width: int, height: int) -> tuple[int, int, int, int]:
    mask = result.masks.data[index].detach().float().cpu().numpy()
    if mask.shape != (height, width):
        mask = cv2.resize(mask, (width, height), interpolation=cv2.INTER_NEAREST)
    ys, xs = np.where(mask > 0.5)
    if len(xs):
        x1, y1, x2, y2 = int(xs.min()), int(ys.min()), int(xs.max()) + 1, int(ys.max()) + 1
    else:
        x1, y1, x2, y2 = map(int, result.boxes.xyxy[index].detach().cpu().tolist())
    pad = int(round(max(x2 - x1, y2 - y1) * PAD_FRACTION))
    return max(0, x1 - pad), max(0, y1 - pad), min(width, x2 + pad), min(height, y2 + pad)


def write_checkpoint(rows: list[dict]) -> pd.DataFrame:
    frame = pd.DataFrame(rows)
    frame = frame.drop_duplicates(subset=["relative_path"], keep="last")
    frame = frame.sort_values("relative_path").reset_index(drop=True)
    frame.to_csv(PREDICTIONS_CSV, index=False)
    return frame


if PREDICTIONS_CSV.exists():
    existing = pd.read_csv(PREDICTIONS_CSV)
    rows = existing.to_dict(orient="records")
    completed = set(existing["relative_path"].astype(str))
    print(f"Resuming: {len(completed):,} images already recorded")
else:
    rows = []
    completed = set()

pending = [p for p in IMAGE_PATHS if p.relative_to(DATASET_ROOT).as_posix() not in completed]
processed_since_checkpoint = 0

for offset in tqdm(range(0, len(pending), BATCH_SIZE), desc="YOLO bottle segmentation"):
    batch_paths = pending[offset : offset + BATCH_SIZE]
    batch_arrays = []
    valid_paths = []

    for path in batch_paths:
        relative = path.relative_to(DATASET_ROOT).as_posix()
        try:
            rgb = load_rgb(path)
            batch_arrays.append(rgb[:, :, ::-1])  # RGB -> BGR for Ultralytics
            valid_paths.append((path, relative, rgb.shape[1], rgb.shape[0]))
        except Exception as error:
            rows.append({
                "relative_path": relative, "status": "decode_error",
                "error": f"{type(error).__name__}: {error}",
            })

    if batch_arrays:
        results = model.predict(
            source=batch_arrays,
            imgsz=IMAGE_SIZE,
            conf=INFERENCE_CONFIDENCE_FLOOR,
            iou=IOU_THRESHOLD,
            classes=[BOTTLE_CLASS_ID],
            retina_masks=True,
            device=DEVICE,
            half=USE_HALF,
            verbose=False,
        )

        for (path, relative, width, height), result in zip(valid_paths, results):
            if result.boxes is None or result.masks is None or len(result.boxes) == 0:
                rows.append({
                    "relative_path": relative, "status": "no_detection",
                    "image_width": width, "image_height": height,
                    "detections_count": 0,
                })
                continue

            confidences = result.boxes.conf.detach().float().cpu().numpy()
            selected = int(np.argmax(confidences))
            confidence = float(confidences[selected])
            x1, y1, x2, y2 = padded_mask_bbox(result, selected, width, height)
            rows.append({
                "relative_path": relative,
                "status": "detected",
                "image_width": width,
                "image_height": height,
                "detections_count": int(len(confidences)),
                "yolo_confidence": confidence,
                "uncertainty": 1.0 - confidence,
                "crop_x1": x1, "crop_y1": y1, "crop_x2": x2, "crop_y2": y2,
            })

    processed_since_checkpoint += len(batch_paths)
    if processed_since_checkpoint >= CHECKPOINT_EVERY:
        write_checkpoint(rows)
        processed_since_checkpoint = 0

predictions = write_checkpoint(rows)
print(predictions["status"].value_counts(dropna=False))
print(f"Saved: {PREDICTIONS_CSV}")

In [ ]:
valid = predictions[predictions["status"] == "detected"].copy()
valid["yolo_confidence"] = pd.to_numeric(valid["yolo_confidence"], errors="coerce")
valid = valid.dropna(subset=["yolo_confidence"])
if len(valid) < LOWEST_N + HIGHEST_N:
    raise RuntimeError(
        f"Need at least {LOWEST_N + HIGHEST_N} valid detections, found {len(valid)}"
    )

least_confident = valid.nsmallest(LOWEST_N, "yolo_confidence").copy()
most_confident = valid.nlargest(HIGHEST_N, "yolo_confidence").copy()
least_confident["group"] = "least_confident"
most_confident["group"] = "most_confident"
least_confident["rank"] = range(1, len(least_confident) + 1)
most_confident["rank"] = range(1, len(most_confident) + 1)
selected = pd.concat([least_confident, most_confident], ignore_index=True)

for directory_name in ("least_confident_30", "most_confident_10"):
    directory = OUTPUT_ROOT / directory_name
    if directory.exists():
        shutil.rmtree(directory)
    directory.mkdir(parents=True)

crop_paths = []
for row_index, row in selected.iterrows():
    source_path = DATASET_ROOT / row["relative_path"]
    rgb = load_rgb(source_path)
    x1, y1, x2, y2 = (int(row[key]) for key in ("crop_x1", "crop_y1", "crop_x2", "crop_y2"))
    crop = Image.fromarray(rgb[y1:y2, x1:x2])
    short_hash = hashlib.sha1(row["relative_path"].encode()).hexdigest()[:8]
    directory_name = "least_confident_30" if row["group"] == "least_confident" else "most_confident_10"
    filename = (
        f"{int(row['rank']):02d}_conf-{float(row['yolo_confidence']):.4f}_"
        f"{short_hash}_{source_path.stem}.jpg"
    )
    destination = OUTPUT_ROOT / directory_name / filename
    crop.save(destination, quality=95, subsampling=0)
    crop_paths.append(destination.relative_to(OUTPUT_ROOT).as_posix())

selected["crop_path"] = crop_paths
selected.to_csv(OUTPUT_ROOT / "selected_crops.csv", index=False)
least_confident = selected[selected["group"] == "least_confident"].sort_values("rank").copy()
most_confident = selected[selected["group"] == "most_confident"].sort_values("rank").copy()
predictions[predictions["status"] == "no_detection"].to_csv(
    OUTPUT_ROOT / "no_detection.csv", index=False
)

display(selected[["group", "rank", "yolo_confidence", "relative_path", "crop_path"]])

In [ ]:
def make_contact_sheet(rows: pd.DataFrame, destination: Path, columns: int = 5) -> Path:
    tile_width, tile_height, caption_height = 280, 360, 64
    sheet_rows = math.ceil(len(rows) / columns)
    sheet = Image.new("RGB", (columns * tile_width, sheet_rows * (tile_height + caption_height)), "white")
    draw = ImageDraw.Draw(sheet)
    font = ImageFont.load_default()

    for position, (_, row) in enumerate(rows.iterrows()):
        crop = Image.open(OUTPUT_ROOT / row["crop_path"]).convert("RGB")
        crop.thumbnail((tile_width - 12, tile_height - 12), Image.Resampling.LANCZOS)
        x0 = (position % columns) * tile_width
        y0 = (position // columns) * (tile_height + caption_height)
        paste_x = x0 + (tile_width - crop.width) // 2
        paste_y = y0 + (tile_height - crop.height) // 2
        sheet.paste(crop, (paste_x, paste_y))
        caption = f"#{int(row['rank'])} conf={float(row['yolo_confidence']):.4f}\n{Path(row['relative_path']).name[:38]}"
        draw.multiline_text((x0 + 6, y0 + tile_height + 4), caption, fill="black", font=font, spacing=3)

    sheet.save(destination, quality=92, subsampling=0)
    return destination


least_sheet = make_contact_sheet(least_confident, OUTPUT_ROOT / "least_confident_30.jpg")
most_sheet = make_contact_sheet(most_confident, OUTPUT_ROOT / "most_confident_10.jpg")

summary = {
    "images_scanned": int(len(predictions)),
    "valid_bottle_detections": int((predictions["status"] == "detected").sum()),
    "no_detection": int((predictions["status"] == "no_detection").sum()),
    "decode_errors": int((predictions["status"] == "decode_error").sum()),
    "least_confident_count": int(len(least_confident)),
    "most_confident_count": int(len(most_confident)),
    "least_confident_range": [
        float(least_confident["yolo_confidence"].min()),
        float(least_confident["yolo_confidence"].max()),
    ],
    "most_confident_range": [
        float(most_confident["yolo_confidence"].min()),
        float(most_confident["yolo_confidence"].max()),
    ],
}
(OUTPUT_ROOT / "summary.json").write_text(json.dumps(summary, indent=2))
archive = shutil.make_archive(str(OUTPUT_ROOT), "zip", root_dir=OUTPUT_ROOT)

print(json.dumps(summary, indent=2))
print(f"Archive: {archive}")
display(Image.open(least_sheet))
display(Image.open(most_sheet))

## Output

Download `/kaggle/working/yolo_bottle_confidence_audit.zip`. It contains the 40 requested whole-bottle crops, both contact sheets, the full prediction table, the selected rows, no-detection rows, run settings, and the summary. The confidence values are raw YOLO instance confidences; they are not calibrated probabilities.